# 파일 형식에 따른 분기 처리

In [8]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day02" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w5" / "day02"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


In [9]:
from app.rag.local_parsers import parse_docx, parse_pdf

NAIVE = {'.docx': parse_docx, '.pdf': parse_pdf}

ok = skipped = 0
empty = []

for f in sorted((p for p in SAMPLES.rglob('*') if p.is_file())):
    handler = NAIVE.get(f.suffix.lower())
    if handler is None:
        skipped += 1
        continue
    doc = handler(f)
    ok += 1
    if not doc.blocks:
        empty.append(f.name)
print(f'훑은 파일    : {ok + skipped}건')
print(f'성공(처리)   : {ok}건')
print(f'건너        : {skipped}건')
print(f'그중 글자 0  : {len(empty)}건 → {empty}')


15:17:32 INFO     app.rag.local_parsers : 텍스트 없음 (스캔본으로 보임): sample_scanned.pdf p.1
훑은 파일    : 30건
성공(처리)   : 21건
건너        : 9건
그중 글자 0  : 1건 → ['sample_scanned.pdf']


In [10]:
from app.rag.local_parsers import parse_hwpx

HANDLERS = {
    ".docx" : parse_docx,
    ".pdf" : parse_pdf,
    ".hwpx" : parse_hwpx
}

print("우리가 읽을 수 있는 형식 : ", list(HANDLERS))
print()

for name in ["규정_v2.0.hwpx", "규정_v2.0.HWPX", "규정_구버전.hwp", "규정_구버전.rtf"]:
    raw = pathlib.Path(name).suffix
    ext = raw.lower() # 소문자는 소문자, 대문자는 대문자 기준잡아서 사용
    handler = HANDLERS.get(ext) # 없어도 keyerror 발생x
    print(f"{name} -> {raw} -> {ext} -> {handler.__name__ if handler else '(없음)'}")

우리가 읽을 수 있는 형식 :  ['.docx', '.pdf', '.hwpx']

규정_v2.0.hwpx -> .hwpx -> .hwpx -> parse_hwpx
규정_v2.0.HWPX -> .HWPX -> .hwpx -> parse_hwpx
규정_구버전.hwp -> .hwp -> .hwp -> (없음)
규정_구버전.rtf -> .rtf -> .rtf -> (없음)


In [11]:
def parse_local_draft(path) :
    p = pathlib.Path(path)
    ext = p.suffix.lower()
    handler = HANDLERS.get(ext)
    if handler is None:
        return None
    return handler(p)

HWPX = next(SAMPLES.glob("DOC-HR-014_*_v2.0.hwpx"))
doc = parse_local_draft(HWPX)
print(f"한글 문서 - 블록 : {len(doc.blocks)}, 표 : {doc.table_count}, 페이지 : {doc.page_count}")

old = pathlib.Path("출장버전_구버전.hwp")
print("구버전 파일 : ", parse_local_draft(old))
print("파일 존재 여부 : ", old.exists())


한글 문서 - 블록 : 233, 표 : 7, 페이지 : 1
구버전 파일 :  None
파일 존재 여부 :  False


In [14]:
import tempfile
from app.core.exceptions import ValidationFailed
_CONVERT_HINT = {
	'.hwp': '한글에서 열고 [다른 이름으로 저장] → HWPX 또는 PDF 로 내보내세요. 구 .hwp 는 한컴 독자 바이너리라 열지 않고는 읽을 방법이 없습니다',
	 '.doc': 'Word 에서 열고 .docx 로 저장하세요', 
	 '.ppt': 'PowerPoint 에서 열고 .pptx 로 저장하세요', 
	 '.xls': 'Excel 에서 열고 .xlsx 로 저장하세요', 
	 '.hwt': '한글 서식 파일입니다. 내용이 든 .hwpx 문서를 넣으세요', 
	 '.zip': '압축을 풀고 안의 문서를 한 건씩 넣으세요'
 }
TMP = pathlib.Path(tempfile.mkdtemp())
hwp = TMP / '출장규정_구버전.hwp'
hwp.write_bytes(b'')
for target in [hwp, TMP / '출장규정_구버전.rtf']:
    ext = target.suffix.lower()
    try:
        if ext in _CONVERT_HINT:
            raise ValidationFailed(f'읽을 수 없는 형식입니다: {ext}', detail=_CONVERT_HINT[ext])
        raise ValidationFailed(f'지원하지 않는 형식입니다: {ext}', detail='핸들러를 추가하면 지원할 수 있습니다')
    except ValidationFailed as exc:
        print(f'[{ext}] {exc}')
        print(f'      안내 → {exc.detail}')


[.hwp] 읽을 수 없는 형식입니다: .hwp
      안내 → 한글에서 열고 [다른 이름으로 저장] → HWPX 또는 PDF 로 내보내세요. 구 .hwp 는 한컴 독자 바이너리라 열지 않고는 읽을 방법이 없습니다
[.rtf] 지원하지 않는 형식입니다: .rtf
      안내 → 핸들러를 추가하면 지원할 수 있습니다


In [15]:
SCANNED = SAMPLES / '_formats' / 'sample_scanned.pdf'
doc = parse_pdf(SCANNED)
print(f'{SCANNED.name} : 블록 {len(doc.blocks)} · 표 {doc.table_count} · 쪽 {doc.page_count}')
print('예외가 났는가 : 아니오 (여기까지 왔다는 것이 곧 증거다)')
print()
try:
    if not doc.blocks:
        raise ValidationFailed(f'문서에서 글자를 찾지 못했습니다: {SCANNED.name}', detail='스캔본(이미지)일 수 있습니다. .env 의 UPSTAGE_PARSE_OCR 를 force 로 두고 실동작 모드로 다시 적재해 보세요')
except ValidationFailed as exc:
    print('막았습니다 :', exc)
    print('안내       :', exc.detail)


15:26:40 INFO     app.rag.local_parsers : 텍스트 없음 (스캔본으로 보임): sample_scanned.pdf p.1
sample_scanned.pdf : 블록 0 · 표 0 · 쪽 1
예외가 났는가 : 아니오 (여기까지 왔다는 것이 곧 증거다)

막았습니다 : 문서에서 글자를 찾지 못했습니다: sample_scanned.pdf
안내       : 스캔본(이미지)일 수 있습니다. .env 의 UPSTAGE_PARSE_OCR 를 force 로 두고 실동작 모드로 다시 적재해 보세요


In [16]:
from app.core.logging import get_logger

log = get_logger(__name__)

broken = TMP / "broken.hwpx"
broken.write_bytes(b"")

try:
    result = parse_hwpx(broken)
except Exception as e:
    log.warning("로컬 파싱 실패 (%s): %s", broken.name, e)
    result = None

print(result)

15:26:50 WARNING  __main__ : 로컬 파싱 실패 (broken.hwpx): File is not a zip file
None


In [17]:
path = ROOT / 'backend' / 'app' / 'rag' / 'local_parsers.py'
text = path.read_text(encoding='utf-8')
print(path.name, '·', len(text.splitlines()), '줄')
for name in ['parse_pdf', 'parse_docx', 'parse_hwpx', 'parse_local']:
    print(f'  def {name:<19}:', text.count(f'def {name}('), '개')
print('  HANDLERS 형식           :', text.count(': parse_'), '개')

local_parsers.py · 156 줄
  def parse_pdf          : 1 개
  def parse_docx         : 1 개
  def parse_hwpx         : 1 개
  def parse_local        : 1 개
  HANDLERS 형식           : 3 개


In [18]:
import importlib
from app.rag import local_parsers
importlib.reload(local_parsers)
parse_local = local_parsers.parse_local
DOCX = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))
PDF = next(SAMPLES.glob('DOC-HR-014_*_v2.0.pdf'))
upper = TMP / (HWPX.stem + '.HWPX')
upper.write_bytes(HWPX.read_bytes())
(TMP / '구버전.doc').write_bytes(b'')
(TMP / '구버전.rtf').write_bytes(b'')
CASES = [('한글 문서 재료', HWPX), ('워드 재료', DOCX), ('PDF 재료', PDF), ('대문자 확장자', upper), ('빈 .hwp', hwp), ('빈 .doc', TMP / '구버전.doc'), ('빈 .rtf', TMP / '구버전.rtf'), ('스캔본', SCANNED), ('깨진 .hwpx', broken)]
for label, target in CASES:
    try:
        parsed = parse_local(target)
    except ValidationFailed as exc:
        print(f'{label} → 막힘 : {exc}')
        continue
    if parsed is None:
        print(f'{label} → None (로그를 위에서 보세요)')
    else:
        print(f'{label} → 읽음 : 블록 {len(parsed.blocks)} · 표 {parsed.table_count} · 쪽 {parsed.page_count}')


한글 문서 재료 → 읽음 : 블록 233 · 표 7 · 쪽 1
워드 재료 → 읽음 : 블록 233 · 표 7 · 쪽 1
PDF 재료 → 읽음 : 블록 16 · 표 0 · 쪽 16
대문자 확장자 → 읽음 : 블록 233 · 표 7 · 쪽 1
빈 .hwp → 막힘 : 읽을 수 없는 형식입니다: .hwp
빈 .doc → 막힘 : 읽을 수 없는 형식입니다: .doc
빈 .rtf → 막힘 : 지원하지 않는 형식입니다: .rtf
15:26:55 INFO     app.rag.local_parsers : 텍스트 없음 (스캔본으로 보임): sample_scanned.pdf p.1
스캔본 → 막힘 : 문서에서 글자를 찾지 못했습니다: sample_scanned.pdf
15:26:55 WARNING  app.rag.local_parsers : 로컬 파싱 실패 (broken.hwpx): File is not a zip file
깨진 .hwpx → None (로그를 위에서 보세요)
